# Merge datasets

In [1]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely import Polygon

In [2]:
def create_square_polygon(e, n, size):
    """create square polygon of size ^ 2 meteres
    Args:
        e: E coordinates
        n: N coordinates
        size: Length and width of the polygon. Defaults to 100.
    """
    return Polygon([(e, n), (e + size, n), (e + size, n + size), (e, n + size)])

### load data

In [3]:
data_folder = os.path.join("data")

statpop_filename = os.path.join(data_folder, "ag-b-00.03-vz2022statpop", "STATPOP2022.csv")
statpop = pd.read_csv(statpop_filename, sep=';', encoding='utf-8', usecols=["E_KOORD", "N_KOORD", "RELI", "B22BTOT"])

statent_filename = os.path.join(data_folder, "ag-b-00.03-22-STATENT2022", "STATENT_2022.csv")
statent = pd.read_csv(statent_filename, sep=';', encoding='utf-8', usecols=["ERHJAHR", "E_KOORD", "N_KOORD", "RELI", "B08T", "B08EMPT", "B08VZAT"])

LAUSANNE_HULL_WKT = (
    "POLYGON ((6.6973149 46.4991489, 6.5474464 46.508319, 6.5441708 46.5091808, "
    "6.5429015 46.5134131, 6.5361596 46.5588201, 6.5949948 46.5926757, "
    "6.6582251 46.5991119, 6.7121747 46.571259, 6.7180473 46.5236707, "
    "6.7159339 46.5179337, 6.7123702 46.5099251, 6.7077559 46.5021425, "
    "6.7016307 46.4999401, 6.6973149 46.4991489))"
)

lausanne = gpd.GeoDataFrame(geometry=gpd.GeoSeries.from_wkt([LAUSANNE_HULL_WKT]), crs="EPSG:4326")

nodes_filename = os.path.join(data_folder, "lausanne_drive_nodes.gpkg")
nodes = gpd.read_file(nodes_filename, columns=["osmid", "geometry"])

common_crs = "EPSG:2056"

### transform data

In [4]:
statpop_statent = statpop.merge(statent, on=["RELI", "N_KOORD", "E_KOORD"], how="outer", suffixes=("_statpop", "_statent"))

In [5]:
statpop_statent["geometry"] = statpop_statent.apply(lambda row: create_square_polygon(row["E_KOORD"], row["N_KOORD"], 100), axis=1)

statpop_statent = gpd.GeoDataFrame(statpop_statent, geometry="geometry", crs=common_crs)

In [6]:
pop_ent_laus = gpd.sjoin(statpop_statent, lausanne.to_crs(common_crs), how="inner", predicate="within")
pop_ent_laus.drop(columns=["index_right"], inplace=True)

### Join into one geodataframe

In [7]:
# join each row in statpop_statent to the nearest node
nodes_data = nodes.to_crs(common_crs).sjoin_nearest(pop_ent_laus, how="right", distance_col="distance_to_node")

In [ ]:
last_revision = nodes_data["ERHJAHR"].max()
nodes_data = nodes_data[nodes_data["ERHJAHR"] == last_revision]

In [9]:
nodes_data.drop(columns=["RELI", "E_KOORD", "N_KOORD", "distance_to_node", "ERHJAHR"], inplace=True)

In [ ]:
nodes_data = nodes_data.dissolve(by="osmid", aggfunc="sum")

In [ ]:
nodes_data["population_percentile"] = nodes_data["B22BTOT"].rank(pct=True)
nodes_data["workers_percentile"] = nodes_data["B08EMPT"].rank(pct=True)
nodes_data["node_coef"] = (nodes_data["population_percentile"] + nodes_data["workers_percentile"]) / 2

In [14]:
nodes_data.head()

,geometry,index_left,B22BTOT,B08T,B08EMPT,B08VZAT,population_percentile
osmid,,,,,,,
172251,"POLYGON ((2534600 1152800, 2534700 1152800, 25...",0,112.0,0.0,0.0,0.000000,0.718393
172276,"POLYGON ((2540700 1151100, 2540800 1151100, 25...",1,164.0,5.0,5.0,4.000000,0.812480
280588,"POLYGON ((2536700 1152000, 2536600 1152000, 25...",8,508.0,11.0,21.0,18.781355,0.982522
280615,"POLYGON ((2537400 1152500, 2537500 1152500, 25...",11,10.0,4.0,10.0,7.891768,0.187847
280650,"POLYGON ((2537900 1152700, 2537900 1152800, 25...",28,380.0,154.0,1839.0,1402.330704,0.959980


In [16]:
nodes_data[nodes_data["B22BTOT"] == nodes_data["B22BTOT"].min()]

,geometry,index_left,B22BTOT,B08T,B08EMPT,B08VZAT,population_percentile
osmid,,,,,,,
280662,"POLYGON ((2538300 1152200, 2538400 1152200, 25...",18,0.0,10.0,94.0,71.934750,0.029239
505192,"POLYGON ((2533300 1155000, 2533400 1155000, 25...",36,0.0,9.0,43.0,36.069355,0.029239
505196,"POLYGON ((2533500 1155100, 2533600 1155100, 25...",37,0.0,4.0,26.0,25.700992,0.029239
505211,"MULTIPOLYGON (((2533500 1155300, 2533400 11553...",117,0.0,12.0,168.0,163.346753,0.029239
505215,"POLYGON ((2533200 1155700, 2533300 1155700, 25...",40,0.0,4.0,49.0,35.738077,0.029239
...,...,...,...,...,...,...,...
12487884033,"POLYGON ((2535600 1153000, 2535600 1153100, 25...",9434,0.0,8.0,20.0,16.718517,0.029239
12499075124,"POLYGON ((2535000 1153600, 2535000 1153700, 25...",9436,0.0,8.0,12.0,11.603963,0.029239
12814703868,"POLYGON ((2533100 1155400, 2533200 1155400, 25...",4756,0.0,6.0,749.0,472.888345,0.029239
